In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations
from pandas.api.types import is_integer_dtype
from collections import Counter, defaultdict
from math import ceil

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
# Define the path to the processed data based on the current user
current_user = os.getlogin()

# Define output directory (notebook is in ./notebooks; data is ../data)
DATA_DIR     = Path(rf"/home/{current_user}/local-share")
OUT_DIR      = DATA_DIR / "processed"

missing_values_path = OUT_DIR / "missing_values_imputed.csv"

# Let pandas detect the delimiter
data = pd.read_csv(missing_values_path, sep=None, engine="python", encoding="utf-8-sig")
print(data.shape)
data

In [ ]:
data.columns

# Feature Derivation

In [ ]:
# -------------------------------------------------------------------------
# FEATURE DERIVATION PLAN
# -------------------------------------------------------------------------
# This section outlines the 12 new features to be derived from the current dataset.
# For each feature, we describe:
#   • What it represents and why it’s relevant.
#   • The columns used for calculation.
#   • The formula or logic to compute it.
# -------------------------------------------------------------------------

# =========================================================================
# 1. performance_diff_blocks
# -------------------------------------------------------------------------
# Meaning:
#   Difference in study performance between Block B and Block A.
#   A positive value means improvement, while negative means decline.
#
# Calculation:
#   performance_diff_blocks = Percentage_Credits_B - Percentage_Credits_A
#
# Columns used:
#   - sdo6_results_Percentage_Credits_B
#   - sdo6_results_Percentage_Credits_A
# =========================================================================


# =========================================================================
# 2. change_in_distance
# -------------------------------------------------------------------------
# Meaning:
#   Change in distance (in km) between previous school location and current address
#   relative to the main university campus (postal code 3584_CS).
#
# Calculation:
#   change_in_distance = distance_current - distance_previous
#
# Columns used:
#   - sdo1_postal_distance_distance_to_3584_CS
#   - sdo1_prev_distance_distance_to_3584_CS
# =========================================================================


# =========================================================================
# 3. moved_for_study
# -------------------------------------------------------------------------
# Meaning:
#   Binary indicator of whether a student moved significantly for their studies.
#   Moving is defined as a change in distance greater than a certain threshold (e.g., 5 km).
#
# Calculation:
#   moved_for_study = 1 if abs(change_in_distance) > threshold else 0
#
# Columns used:
#   - Derived column: change_in_distance
# =========================================================================


# =========================================================================
# 4. age_category
# -------------------------------------------------------------------------
# Meaning:
#   Categorical grouping of students by age at the start of their degree.
#   Helps capture non-linear age effects.
#
# Suggested bins:
#   <21 years, 21–25 years, >25 years
#
# Calculation:
#   Based on sdo1_characteristics_age_start_study using pd.cut().
#
# Columns used:
#   - sdo1_characteristics_age_start_study
# =========================================================================


# =========================================================================
# 5. time_first_orient_to_start
# -------------------------------------------------------------------------
# Meaning:
#   Number of days between the first orientation event and the official degree start date.
#   Measures how early a student started engaging with the university.
#
# Calculation:
#   (degree_start_date - first_orientation_date).days
#
# Columns used:
#   - sdo5_degree_degree_start_date
#   - sdo2_orientation_First_Event_Date
# =========================================================================


# =========================================================================
# 6. time_last_orient_to_start
# -------------------------------------------------------------------------
# Meaning:
#   Number of days between the last orientation event and the degree start date.
#   Smaller values indicate participation close to enrollment.
#
# Calculation:
#   (degree_start_date - last_orientation_date).days
#
# Columns used:
#   - sdo5_degree_degree_start_date
#   - sdo2_orientation_Last_Event_Date
# =========================================================================


# =========================================================================
# 7. gap_skc_to_start
# -------------------------------------------------------------------------
# Meaning:
#   Time gap (in days) between the SKC interview/advice and the degree start date.
#   Reflects how long before starting the student engaged with SKC.
#
# Calculation:
#   (degree_start_date - skc_date).days
#
# Columns used:
#   - sdo5_degree_degree_start_date
#   - sdo2_skc_SKC_DATUM
# =========================================================================


# =========================================================================
# 8. time_enroll_to_start
# -------------------------------------------------------------------------
# Meaning:
#   Time difference between the formal enrollment date and the start of the degree.
#   Indicates administrative or preparatory delay before classes begin.
#
# Calculation:
#   (degree_start_date - enrollment_date).days
#
# Columns used:
#   - sdo5_degree_degree_start_date
#   - sdo5_degree_enrollment_date
# =========================================================================


# =========================================================================
# 9. has_open_day
# -------------------------------------------------------------------------
# Meaning:
#   Indicates whether the student attended any Open Day–related orientation activity.
#
# Calculation:
#   1 if Event_Types_Attended in ['Open_Day_Related', 'Multiple_Types'] else 0
#
# Columns used:
#   - sdo2_orientation_Event_Types_Attended
# =========================================================================


# =========================================================================
# 10. has_proefstuderen
# -------------------------------------------------------------------------
# Meaning:
#   Indicates whether the student participated in a Trial or Experience Day
#   (Proefstuderen, Dutch equivalent).
#
# Calculation:
#   1 if Event_Types_Attended in ['Trial_or_Experience_Day', 'Multiple_Types'] else 0
#
# Columns used:
#   - sdo2_orientation_Event_Types_Attended
# =========================================================================


# =========================================================================
# 11. has_campus_tour
# -------------------------------------------------------------------------
# Meaning:
#   Indicates whether the student attended a campus tour event.
#
# Calculation:
#   1 if Event_Types_Attended == 'Campus_Tour' else 0
#
# Columns used:
#   - sdo2_orientation_Event_Types_Attended
# =========================================================================


# =========================================================================
# 12. has_any_orientation
# -------------------------------------------------------------------------
# Meaning:
#   Captures whether the student attended *any* orientation event at all.
#   Students with 'Absent' are coded as 0, others as 1.
#
# Calculation:
#   1 if Event_Types_Attended != 'Absent' else 0
#
# Columns used:
#   - sdo2_orientation_Event_Types_Attended
# =========================================================================


In [ ]:
data[['sdo2_skc_SKC_DATUM', 'sdo5_degree_degree_start_date']].head(10)

In [ ]:
# ------------------------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------------------------
# Derive 12 new features from the current dataset. All time deltas are in WEEKS.
#
# New columns created:
#   1) performance_diff_blocks                    (float)
#   2) change_in_distance                         (float, km)
#   3) moved_for_study                            (int, {0,1})
#   4) age_category                               (category: '<21', '21–25', '>25')
#   5) time_first_orient_to_start_weeks           (float, weeks)
#   6) time_last_orient_to_start_weeks            (float, weeks)
#   7) gap_skc_to_start_weeks                     (float, weeks)
#   8) time_enroll_to_start_weeks                 (float, weeks)
#   9) has_open_day                               (int, {0,1})
#  10) has_proefstuderen                          (int, {0,1})
#  11) has_campus_tour                            (int, {0,1})
#  12) has_any_orientation                        (int, {0,1})
#
# Notes:
# - Uses 'sdo5_degree_start_date' (the fixed canonical column).
# - Uses 'weeks' for all derived time differences (original columns untouched).
# - Robust to missing/invalid dates and numerics (coerced to NaN).
# - Threshold for moved_for_study is configurable (default: 5 km).
# ------------------------------------------------------------------------------

def _get_dt(df, col):
    s = df.get(col)
    if s is None:
        return pd.Series(pd.NaT, index=df.index)
    return pd.to_datetime(s, errors='coerce')

def _get_num(df, col):
    s = df.get(col)
    if s is None:
        return pd.Series(np.nan, index=df.index)
    return pd.to_numeric(s, errors='coerce')

def derive_features(df, moved_threshold_km: float = 5.0,
                    age_bins=None, age_labels=None, rounding=2):
    """
    Derive the 12 requested features. Returns a new DataFrame (copy).
    Robust to missing columns: missing dates → NaT; missing numerics → NaN.
    """
    df_out = df.copy()

    # ---------- Ensure numeric inputs ----------
    perc_A    = _get_num(df_out, 'sdo6_results_Percentage_Credits_A')
    perc_B    = _get_num(df_out, 'sdo6_results_Percentage_Credits_B')
    dist_curr = _get_num(df_out, 'sdo1_postal_distance_distance_to_3584_CS')
    dist_prev = _get_num(df_out, 'sdo1_prev_distance_distance_to_3584_CS')
    age_start = _get_num(df_out, 'sdo1_characteristics_age_start_study')

    # ---------- Dates (parsed) ----------
    deg_start    = _get_dt(df_out, 'sdo5_degree_start_date')  # canonical fixed column
    first_orient = _get_dt(df_out, 'sdo2_orientation_First_Event_Date')
    last_orient  = _get_dt(df_out, 'sdo2_orientation_Last_Event_Date')
    skc_date     = _get_dt(df_out, 'sdo2_skc_SKC_DATUM')
    enroll_date  = _get_dt(df_out, 'sdo5_degree_enrollment_date')

    # ---------- 1) performance_diff_blocks ----------
    df_out['performance_diff_blocks'] = (perc_B - perc_A).round(rounding)

    # ---------- 2) change_in_distance ----------
    df_out['change_in_distance'] = (dist_curr - dist_prev).round(rounding)

    # ---------- 3) moved_for_study ----------
    df_out['moved_for_study'] = (
        df_out['change_in_distance'].abs() > moved_threshold_km
    ).astype('Int64')

    # ---------- 4) age_category ----------
    if age_bins is None:
        age_bins = [-np.inf, 21, 25, np.inf]
    if age_labels is None:
        age_labels = ['<21', '21–25', '>25']
    df_out['age_category'] = pd.cut(
        age_start, bins=age_bins, labels=age_labels, right=True, include_lowest=True
    )

    # ---------- Week-based deltas ----------
    df_out['time_first_orient_to_start_weeks'] = _weeks_between(deg_start, first_orient).round(rounding)
    df_out['time_last_orient_to_start_weeks']  = _weeks_between(deg_start, last_orient).round(rounding)
    df_out['gap_skc_to_start_weeks']           = _weeks_between(deg_start, skc_date).round(rounding)
    df_out['time_enroll_to_start_weeks']       = _weeks_between(deg_start, enroll_date).round(rounding)

    # ---------- Event-type flags ----------
    evt = df_out.get('sdo2_orientation_Event_Types_Attended',
                     pd.Series(index=df_out.index, dtype='object')).fillna('')
    evt_norm = evt.astype(str).str.strip()

    df_out['has_open_day']        = evt_norm.isin(['Open_Day_Related', 'Multiple_Types']).astype('Int64')
    df_out['has_proefstuderen']   = evt_norm.isin(['Trial_or_Experience_Day', 'Multiple_Types']).astype('Int64')
    df_out['has_campus_tour']     = evt_norm.eq('Campus_Tour').astype('Int64')
    df_out['has_any_orientation'] = (~evt_norm.eq('Absent')).astype('Int64')

    return df_out
data = derive_features(data, moved_threshold_km=5.0, rounding=2)

# Preview the derived numerical and date-related columns
display(
    data.filter(regex="^sdo5_degree_start_date$|performance_diff_blocks|change_in_distance|moved_for_study|age_category|_weeks$|^has_").head(15)
)

# Optional quick stats to check ranges
print("\nSummary of time deltas (in weeks):")
print(data.filter(regex="_weeks$").describe().T)


In [ ]:
# ------------------------------------------------------------------------------
# INTERPRETATION OF THE ABOVE WEEK-BASED DERIVATIONS
# ------------------------------------------------------------------------------
# The derived time-delta features in weeks appear realistic and consistent with
# expected student timelines, confirming that the repaired start-date column is
# functioning correctly.
#
# • The average time between the first orientation and the degree start date is
#   about 33 weeks (roughly eight months). This indicates that most students
#   begin engaging with the university well in advance of starting their studies.
#
# • The time between the last orientation and the start date averages around
#   31 weeks (about seven months), which is slightly shorter than for the first
#   orientation. This pattern makes sense because the final orientation event
#   typically happens closer to the beginning of the degree.
#
# • The gap between the SKC date and the start date has an average of about
#   10 weeks but shows a wide spread, with some negative values and a large
#   standard deviation. Positive values mean the SKC occurred before the start
#   (as expected), while negative values indicate that it was held after the
#   start or recorded late in the system. This variation will later be captured
#   in a categorical version of the feature.
#
# • The time between enrollment and the start date averages roughly 2,667 weeks,
#   which is about 51 years. This value is clearly unrealistic and suggests that
#   the enrollment date field in the raw data is not a true date but possibly a
#   placeholder or misinterpreted value. This feature should be inspected or
#   excluded from analysis until verified.
#
# Overall, the orientation-related time differences look credible and confirm
# that the start-date repair logic worked properly. The SKC and enrollment
# differences highlight where further data cleaning or interpretation may be
# necessary.
# ------------------------------------------------------------------------------


# DERIVING NEW CATEGORICAL FEATURES 


In [ ]:
# ------------------------------------------------------------------------------
# INTERPRETING NEGATIVE VALUES AND DERIVING NEW CATEGORICAL FEATURES
# ------------------------------------------------------------------------------
# In the previous step, we derived several numerical features expressed as differences.
# Some of these difference-based features (e.g., gap_skc_to_start_weeks, change_in_distance,
# performance_diff_blocks) can take negative values. These negative values are not
# necessarily "errors" — they carry meaningful directional information.
#
# Below we explain how to interpret these and how to derive new categorical features
# to capture their qualitative meaning.
# ------------------------------------------------------------------------------

# =========================================================================
# 1. gap_skc_to_start_weeks → SKC timing interpretation
# -------------------------------------------------------------------------
# • Positive values: SKC happened BEFORE the degree start date (expected).
# • Negative values: SKC happened AFTER the degree start date (unusual or delayed).
#
# New categorical feature:
#   skc_timing_category:
#       - 'skc_happened_earlier'  → gap_skc_to_start_weeks > 0
#       - 'skc_happened_late'     → gap_skc_to_start_weeks < 0
# =========================================================================


# =========================================================================
# 2. change_in_distance → Movement interpretation
# -------------------------------------------------------------------------
# • Positive values: The student’s current residence is FARTHER from the university
#   compared to their previous school — i.e., they moved away.
# • Negative values: The student moved CLOSER to the university.
#
# To make this interpretation intuitive, we will assign:
#   - 'moved_closer'  → change_in_distance < 0
#   - 'moved_farther' → change_in_distance > 0
#
# New categorical feature:
#   distance_change_category
# =========================================================================


# =========================================================================
# 3. performance_diff_blocks → Academic trend interpretation
# -------------------------------------------------------------------------
# • Positive values: Performance improved from Block A to Block B.
# • Negative values: Performance declined between the two blocks.
#
# New categorical feature:
#   performance_trend_category:
#       - 'improved'  → performance_diff_blocks > 0
#       - 'reduced'  → performance_diff_blocks < 0
#
# Students with 0 difference can be labeled as 'no_change'.
# =========================================================================

# These new categorical versions will help in:
#   • Exploratory analysis (clearer visuals and summaries).
#   • Modeling (e.g., one-hot encoding of trends and directions).
#   • Interpretability of student behavior and academic patterns.
# ------------------------------------------------------------------------------

In [ ]:
# ------------------------------------------------------------------------------
# CREATE CATEGORICAL VERSIONS OF THREE DIRECTIONAL FEATURES
# ------------------------------------------------------------------------------
# Converts three numeric "difference" features into categorical interpretations.
# Assumes these numeric features already exist from the derivation step:
#   - gap_skc_to_start_weeks
#   - change_in_distance
#   - performance_diff_blocks
#
# Robustness:
# - Validates required columns exist.
# - Preserves missing values as <NA> in the categorical outputs.
# - Uses vectorized mapping via np.select for clarity and speed.
# ------------------------------------------------------------------------------

def _require_cols(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(
            f"Missing required column(s): {missing}. "
            "Run the derivation step first to create them."
        )

_required = [
    'gap_skc_to_start_weeks',
    'change_in_distance',
    'performance_diff_blocks',
]
_require_cols(data, _required)

# ---------------------------
# 1) SKC timing category
# ---------------------------
skc = data['gap_skc_to_start_weeks']
data['skc_timing_category'] = pd.Series(
    np.select(
        condlist=[skc > 0, skc < 0, skc == 0],
        choicelist=['skc_happened_earlier', 'skc_happened_late', 'no_gap'],
        default=pd.NA
    ),
    index=data.index,
    dtype='object'
).astype('category')

# ---------------------------
# 2) Distance change category
# ---------------------------
# NOTE: per your requested interpretation:
#   > 0 → 'moved_closer'
#   < 0 → 'moved_farther'
#   = 0 → 'no_change'
dist = data['change_in_distance']
data['distance_change_category'] = pd.Series(
    np.select(
        condlist=[dist > 0, dist < 0, dist == 0],
        choicelist=['moved_closer', 'moved_farther', 'no_change'],
        default=pd.NA
    ),
    index=data.index,
    dtype='object'
).astype('category')

# ---------------------------
# 3) Performance trend category
# ---------------------------
#   > 0 → 'improved'
#   < 0 → 'reduced'
#   = 0 → 'no_change'
perf = data['performance_diff_blocks']
data['performance_trend_category'] = pd.Series(
    np.select(
        condlist=[perf > 0, perf < 0, perf == 0],
        choicelist=['improved', 'reduced', 'no_change'],
        default=pd.NA
    ),
    index=data.index,
    dtype='object'
).astype('category')

# ---------------------------
# Optional: quick sanity checks
# ---------------------------
print("skc_timing_category:\n", data['skc_timing_category'].value_counts(dropna=False))
print("\ndistance_change_category:\n", data['distance_change_category'].value_counts(dropna=False))
print("\nperformance_trend_category:\n", data['performance_trend_category'].value_counts(dropna=False))

In [ ]:
data[['sdo2_skc_SKC_DATUM', 'sdo5_degree_start_date']].head(10)

In [ ]:
data.columns

# Concern Over Duplicates after feature derivation

In [ ]:

# ---------------------------
# Check Possible duplicate columns
# ---------------------------

# 1) Check if the two IDs are the same
same_id = (data['SINH_ID'].astype(str).str.strip()
           == data['sdo2_orientation_STUDENTNUMMER'].astype(str).str.strip())
print("Student ID equality rate:", same_id.mean())

# 2) Check if previous-school postcode columns match
if 'sdo1_prev_distance_postcode' in data and 'sdo1_previous_Previous_School_Postal_Code' in data:
    same_prev_pc = (data['sdo1_prev_distance_postcode'].astype(str).str.strip()
                    == data['sdo1_previous_Previous_School_Postal_Code'].astype(str).str.strip())
    print("Prev-school postcode equality rate:", same_prev_pc.mean())

# 3) Orientation presence redundancy
if 'has_orientation' in data and 'has_any_orientation' in data:
    print("has_orientation vs has_any_orientation agreement:",
          (data['has_orientation'].astype('Int64') == data['has_any_orientation'].astype('Int64')).mean())

# 4) SKC presence redundancy
if 'has_skc' in data and 'sdo2_skc_SKC_DATUM_missing_flag' in data:
    # If missing flag means "date missing", invert to compare to has_skc.
    inv_missing = 1 - data['sdo2_skc_SKC_DATUM_missing_flag'].astype('Int64')
    print("has_skc vs not-missing SKC date agreement:",
          (data['has_skc'].astype('Int64') == inv_missing).mean())


In [ ]:
# ------------------------------------------------------------------------------
# CLEANUP: DROP CONFIRMED DUPLICATE / REDUNDANT COLUMNS
# ------------------------------------------------------------------------------
# This cell drops columns we validated as redundant based on your checks:
#   • Keep SINH_ID as primary student key; drop sdo2_orientation_STUDENTNUMMER.
#   • Keep sdo1_prev_distance_postcode (used by distance features);
#     drop sdo1_previous_Previous_School_Postal_Code.
#   • Keep has_any_orientation (derived from actual events); drop has_orientation.
#   • Keep has_skc; drop sdo2_skc_SKC_DATUM_missing_flag.
#   • Keep sdo5_degree_start_date (canonical, repaired); drop sdo5_degree_degree_start_date.
#
# Optional: set DROP_ALL_MISSING_FLAGS=True to remove every '*_missing_flag' column.
# Everything is reported, and dropped columns are archived in `dropped_columns_archive`.
# ------------------------------------------------------------------------------

import pandas as pd
import re

# --- Configuration toggles ---
DROP_ALL_MISSING_FLAGS = False   # set to True if you want to drop ALL *_missing_flag columns

# --- Columns we chose to drop (confirmed by your diagnostics) ---
candidate_drop = [
    'sdo2_orientation_STUDENTNUMMER',
    'sdo1_previous_Previous_School_Postal_Code',
    'has_orientation',
    'sdo2_skc_SKC_DATUM_missing_flag',
    'sdo5_degree_degree_start_date',
]

# --- Optionally add ALL *_missing_flag columns to the drop list ---
if DROP_ALL_MISSING_FLAGS:
    missing_flag_cols = [c for c in data.columns if c.endswith('_missing_flag')]
    # Ensure we don't double add already-listed items
    candidate_drop = list(dict.fromkeys(candidate_drop + missing_flag_cols))

# --- Compute which of these actually exist in the DataFrame ---
to_drop = [c for c in candidate_drop if c in data.columns]

print("Columns requested to drop:", len(candidate_drop))
print("Columns found and will be dropped:", len(to_drop))
for c in to_drop:
    print("  -", c)

# --- Archive dropped columns (for safety / audit) ---
dropped_columns_archive = data[to_drop].copy() if to_drop else pd.DataFrame(index=data.index)
print("\nArchived a copy of columns to be dropped in `dropped_columns_archive`.")

# --- Drop and report shape change ---
before_shape = data.shape
data = data.drop(columns=to_drop, errors='ignore')
after_shape = data.shape

print(f"\nShape before: {before_shape} → after: {after_shape}")
print("Done. Kept primary keys and canonical columns:")
keep_examples = [
    'SINH_ID',
    'sdo5_degree_start_date',
    'sdo1_prev_distance_postcode',
    'has_any_orientation',
    'has_skc'
]
print([c for c in keep_examples if c in data.columns])


In [ ]:
# ------------------------------------------------------------------------------
# INSPECT: COLUMNS WITH '_missing_flag'
# ------------------------------------------------------------------------------
# This cell identifies all columns that serve as detailed missingness indicators.
# These columns often duplicate the simpler 'has_*' flags and can be removed safely.
# ------------------------------------------------------------------------------

missing_flag_cols = [c for c in data.columns if c.endswith('_missing_flag')]

print(f"Total '_missing_flag' columns found: {len(missing_flag_cols)}\n")

# Optionally group them by their base prefix (everything before the last underscore)
from collections import defaultdict
grouped_flags = defaultdict(list)
for col in missing_flag_cols:
    base = col.split('_missing_flag')[0]
    group_root = base.split('_')[0]  # rough grouping by sdo1/sdo2/sdo6 etc.
    grouped_flags[group_root].append(col)

for group, cols in grouped_flags.items():
    print(f"--- {group.upper()} ({len(cols)} columns) ---")
    for c in cols:
        print("  ", c)
    print()

# If you prefer a simple list instead of grouped view:
# print(missing_flag_cols)

In [ ]:
# ------------------------------------------------------------------------------
# SUMMARY: POTENTIAL DUPLICATE OR CONCEPTUALLY OVERLAPPING COLUMNS
# ------------------------------------------------------------------------------
# After deriving new features, several columns now express the *same concept*
# in different forms (numeric, categorical, or binary). These are not strict
# duplicates, but they carry overlapping information and should not all be
# used together in modeling.
#
# The main groups of conceptual overlap are:
#
# 1️⃣ Academic Performance
#     - Base: sdo6_results_Percentage_Credits_A / B
#     - Derived numeric: performance_diff_blocks  (B - A)
#     - Derived categorical: performance_trend_category  ('improved', 'declined', 'no_change')
#     → Keep either the numeric or categorical form (not both).
#
# 2️⃣ Residence and Distance
#     - Base: sdo1_prev_distance_distance_to_3584_CS, sdo1_postal_distance_distance_to_3584_CS
#     - Derived numeric: change_in_distance
#     - Derived categorical: distance_change_category  ('moved_closer', 'moved_farther', 'no_change')
#     - Binary summary: moved_for_study  (based on >5 km threshold)
#     → These all describe movement; keep one representation.
#
# 3️⃣ SKC Timing
#     - Base: sdo2_skc_SKC_DATUM, sdo5_degree_start_date
#     - Derived numeric: gap_skc_to_start_weeks
#     - Derived categorical: skc_timing_category  ('skc_happened_earlier', 'skc_happened_late', 'no_gap')
#     → Numeric and categorical forms describe the same time gap.
#
# 4️⃣ Orientation and Engagement
#     - Base: sdo2_orientation_Event_Types_Attended
#     - Derived binaries: has_open_day, has_proefstuderen, has_campus_tour, has_any_orientation
#     → All stem from the same raw event-type column; choose one set.
#
# 5️⃣ Age and Demographics
#     - Base: sdo1_characteristics_age_start_study
#     - Derived categorical: age_category  ('<21', '21–25', '>25')
#     → Same information at different granularity.
#
# 6️⃣ Date Relationships
#     - Base: sdo5_degree_degree_start_date (raw) vs sdo5_degree_start_date (fixed)
#     → Keep only the repaired column: sdo5_degree_start_date.
#
# Additional note:
#   - *_missing_flag columns duplicate the simpler has_* indicators (e.g., has_skc, has_previous).
#     These can be safely removed if you only need a single presence indicator.
#
# ⚙️ Recommendation:
#   • For modeling: prefer numeric versions (continuous features).
#   • For reporting/interpretation: prefer categorical or binary flags.
#   • Remove redundant variants to reduce multicollinearity and simplify the dataset.
# ------------------------------------------------------------------------------


In [ ]:
# ------------------------------------------------------------------------------
# CLEANUP VERSION 1: KEEP NUMERIC FEATURES (drop categorical/binary duplicates)
# ------------------------------------------------------------------------------
# Creates `data_numeric_pref` — a version of the dataset prioritizing numeric
# representations for modeling. Categorical or binary duplicates are dropped.
# ------------------------------------------------------------------------------

data_numeric_pref = data.copy()

drop_for_numeric_pref = [
    # Drop categorical or binary equivalents
    'performance_trend_category',    # categorical version of performance_diff_blocks
    'distance_change_category',      # categorical version of change_in_distance
    'skc_timing_category',           # categorical version of gap_skc_to_start_weeks
    'moved_for_study',               # binary version of distance_change_category
    'has_open_day', 'has_proefstuderen', 'has_campus_tour', 'has_any_orientation',  # event flags
    'age_category'                   # categorical version of age_start_study
]

# Keep orientation counts and numeric differences
data_numeric_pref = data_numeric_pref.drop(
    columns=[c for c in drop_for_numeric_pref if c in data_numeric_pref.columns],
    errors='ignore'
)

print(f"Numeric-preferred dataset shape: {data_numeric_pref.shape}")
print("Dropped categorical/binary duplicates.")


In [ ]:
# Save data_numeric_pref
data_numeric_pref.to_csv(os.path.join("..", "data", "processed", "features_derived_numeric.csv"), header=True)

In [ ]:
# ------------------------------------------------------------------------------
# CLEANUP VERSION 2: KEEP CATEGORICAL FEATURES (drop numeric duplicates)
# ------------------------------------------------------------------------------
# Creates `data_categorical_pref` — a version of the dataset prioritizing
# categorical or interpretable features for easier visualization and encoding.
# Numeric duplicates are dropped.
# ------------------------------------------------------------------------------

data_categorical_pref = data.copy()

drop_for_categorical_pref = [
    # Drop numeric versions of derived relationships
    'performance_diff_blocks',       # numeric version of performance_trend_category
    'change_in_distance',            # numeric version of distance_change_category
    'gap_skc_to_start_weeks',        # numeric version of skc_timing_category
    'time_first_orient_to_start_weeks',
    'time_last_orient_to_start_weeks',
    'time_enroll_to_start_weeks',    # keep only if temporal numeric detail needed
    'sdo1_characteristics_age_start_study'  # numeric version of age_category
]

data_categorical_pref = data_categorical_pref.drop(
    columns=[c for c in drop_for_categorical_pref if c in data_categorical_pref.columns],
    errors='ignore'
)

print(f"Categorical-preferred dataset shape: {data_categorical_pref.shape}")
print("Dropped numeric duplicates.")


In [ ]:
# Save data_categorical_pref
data_categorical_pref.to_csv(os.path.join("..", "data", "processed", "features_derived_categorical.csv"), header=True)